# Review gp_collab_rxnpredict results

Same structure as DOPE-MURI's `02_review_results.ipynb`, reading runs produced
by this package. Set `RUNS` below to the folder holding the finished tasks. The
first cell exports them into the layout `load_results` reads, then loads them;
nothing here launches training.

Needs `numpy pandas scipy scikit-learn matplotlib` (and `ipywidgets` for the
control panel). **No PyTorch** -- review is CPU-only and model-free.

Put this notebook and `results.py` in the `gp_collab_hazel` folder, next to
`gpc/` and `inputs/`.

In [ ]:
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
from gpc.results import (prepare, load_results, lolo_report, plot_parity,
                     plot_lolo_comparison, grouped_predictions)

ROOT = Path.cwd().resolve()
RUNS = ROOT / "runs/rxnpredict_v1"
# Named explicitly because runs/ may also hold probe/ or a *_pilot folder.

# Builds RUNS/hazel_export/collected the first time; pass refresh=True after
# adding or rerunning tasks. Methods are relabelled to hazel_gp's names
# (iid_stratified_4 -> iid_matched, kfold_stratified_5 -> kfold) so the
# arguments below match the original notebook. kfold_5 keeps its own name so
# the unstratified fold is never confused with the stratified one; the folds
# themselves are unchanged either way.
EXPORT = prepare(RUNS, bundle=ROOT / "inputs", refresh=False)

tables, collection = load_results(EXPORT)
print("Complete benchmark:", collection["complete"])
print("Completed tasks:", collection["completed_tasks"], "/", collection["expected_tasks"])
display(tables["summary"])

### Inspect predictions

Pick a model, method and held-out ligand; the parity plot and that split's
metrics redraw. Needs `ipywidgets`; skip this cell if it is not installed.

In [ ]:
from gpc.results import review_controls
display(review_controls(tables))

### LOLO reported three ways

`per_ligand` scores each held-out ligand on its own rows, `mean_across_ligands`
averages those, and `pooled_over_folds` concatenates all eight folds and scores
once. Pooled and averaged values are **not** interchangeable: a per-ligand R² is
measured against that ligand's own (smaller) variance, so the averaged figure is
systematically harsher.

In [ ]:
# Start from an empty figure registry. The inline backend re-renders every
# figure still open at the end of a cell, so anything leaked by an earlier
# run of this kernel would be drawn again here on top of this cell's own.
plt.close("all")

METRIC = "r2"
report = lolo_report(tables["predictions"], tables["metrics_by_split"])
display(report[["model", "view", "scope", "n_test", "r2", "rmse", "mae", "kendall_tau"]].round(3))

fig_ligands = plot_lolo_comparison(tables["metrics_by_split"], metric=METRIC)
display(fig_ligands)
plt.close(fig_ligands)

### Compare the three evaluation methods

LOLO against the two in-distribution controls. The stratified 8-fold has LOLO's
exact fold geometry (2688/384) with every ligand present in training, so the gap
between them is the cost of meeting an unseen ligand.

In [ ]:
# Start from an empty figure registry. The inline backend re-renders every
# figure still open at the end of a cell, so anything leaked by an earlier
# run of this kernel would be drawn again here on top of this cell's own.
plt.close("all")

summary = tables["summary"]
pooled = summary[summary.aggregation == "pooled_predictions"]
display(pooled.pivot(index="model", columns="method", values=["r2", "rmse"]).round(3))

from gpc.results import model_label, method_label

# Grouped bars, not a line. The x axis is five unrelated feature sets, so a line
# joining them would imply an ordering and a continuum that do not exist; bars
# also match plot_lolo_comparison above. Heights are the POOLED figures, so they
# do not equal the mean of that model's per-fold bars in the LOLO chart.
table = pooled.pivot(index="model", columns="method", values=METRIC)
table = table.reindex(index=list(dict.fromkeys(pooled.model)))   # config order, not alphabetical
table = table[[m for m in ("lolo", "iid_matched", "kfold") if m in table.columns]]
table.index = [model_label(m) for m in table.index]
table.columns = [method_label(m) for m in table.columns]

unit = " (% yield)" if METRIC in ("rmse", "mae") else ""
ax = table.plot.bar(figsize=(9, 4.5), width=0.78, edgecolor="white", linewidth=0.6,
                    color=["#c0392b", "#5b8db8", "#a8c8e0"])
ax.set(xlabel="", ylabel=f"pooled {METRIC.upper()}{unit}")
ax.set_ylim(0, table.to_numpy().max() * 1.12)
ax.tick_params(axis="x", rotation=15)
ax.legend(title="Evaluation method", fontsize=8, title_fontsize=8, frameon=False,
          loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax.set_title(f"Pooled out-of-fold {METRIC.upper()} by feature section", fontsize=11)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", fontsize=7, padding=2)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.set_axisbelow(True)

fig_methods = ax.figure
fig_methods.tight_layout()
display(fig_methods)
plt.close(fig_methods)

In [ ]:
grouped_iid = grouped_predictions(tables["predictions"], model="selected_2", method="iid_matched")
print("Reference groups:", list(grouped_iid))
# The stratified 8-fold is a single non-repeating partition with every ligand in
# every fold, so it has no per-ligand reference group and returns one group keyed
# "". Use method="lolo" to get the eight held-out ligands instead:
grouped_lolo = grouped_predictions(tables["predictions"], model="selected_2", method="lolo")
print("LOLO groups:", list(grouped_lolo))
# grouped_lolo["SPhos"]["y_pred"] is a list of arrays, one per split.

### Final export

Nothing is written until `SAVE = True`.

In [ ]:
SAVE = False
EXPORT_DIR = ROOT / "exports/local_review_v1"
FIGURE_FORMATS = ["png", "pdf"]
TABLES_TO_SAVE = ["summary", "metrics_by_split", "predictions"]

if SAVE:
    EXPORT_DIR.mkdir(parents=True, exist_ok=False)
    for name in TABLES_TO_SAVE:
        tables[name].to_csv(EXPORT_DIR / f"{name}.csv", index=False)
    report.to_csv(EXPORT_DIR / "lolo_report.csv", index=False)
    figures = {"lolo_by_ligand": fig_ligands, "method_comparison": fig_methods}
    for name, fig in figures.items():
        for fmt in FIGURE_FORMATS:
            fig.savefig(EXPORT_DIR / f"{name}.{fmt}", dpi=300, bbox_inches="tight")
    print("Saved:", EXPORT_DIR)
else:
    print("Preview only. Set SAVE=True when ready.")